# A roller chain on a turning polygon

Predict what happens to speed ripple when the tooth count doubles while the mean conveying speed is kept fixed. Change the first cell, run all cells, and compare speed, timing and displacement.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

pitch = .0127      # m between adjacent roller centres
teeth = 12         # integer sprocket tooth count
rpm = 120.0        # constant drive shaft speed
sensor_phase = .25 # fraction of one engagement interval, 0 to 1
assert int(teeth) == teeth and teeth >= 4 and pitch > 0 and rpm > 0
assert 0 <= sensor_phase <= 1

## Advance by one chord
A roller centre sweeps an angle of $2\pi/z$ over one engagement. Integrate the projection of its circular velocity on the straight strand. The difference in projected position between the interval endpoints has to be one pitch.

In [ ]:
def chain(z, drive_rpm, phase):
    a = np.pi/z
    R = pitch/(2*np.sin(a))
    omega = 2*np.pi*drive_rpm/60
    theta = (2*np.asarray(phase)-1)*a
    advance = R*(np.sin(theta)+np.sin(a))
    velocity = omega*R*np.cos(theta)
    mean = pitch*z*drive_rpm/60
    return advance,velocity,mean

phase = np.linspace(0,1,501)
advance,v,mean = chain(teeth,rpm,phase)
assert np.isclose(advance[-1]-advance[0],pitch)
assert np.isclose(np.sum(np.diff(phase) * (v[:-1] + v[1:]) / 2)/mean,1,rtol=2e-6)
print(f"Mean strand speed {mean:.5f} m/s; tooth-pass frequency {teeth*rpm/60:.1f} Hz")
print(f"Full speed variation / mean = {100*(v.max()-v.min())/mean:.4f}%")
x_obs,v_obs,_ = chain(teeth,rpm,sensor_phase)
print(f"At phase {sensor_phase:.3f}: advance = {1e3*x_obs:.4f} mm, speed = {v_obs:.5f} m/s")

## Compare equal mean speeds
The doubled-tooth sprocket must rotate half as fast to maintain the same pitch throughput. The phase coordinate stays comparable because the number of engagements per second is also unchanged.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(9,3.5),layout="constrained")
for z, r, style in [(teeth,rpm,"-"),(2*teeth,rpm/2,"--")]:
    x,u,m = chain(z,r,phase)
    ax[0].plot(phase,100*(u/m-1),style,label=f"{z} teeth")
    ax[1].plot(phase,(x-phase*pitch)*1e6,style,label=f"{z} teeth")
ax[0].set(xlabel="Fraction of an engagement",ylabel="Speed departure from mean (%)",title="Instantaneous strand speed")
ax[1].set(xlabel="Fraction of an engagement",ylabel="Position error (µm)",title="Relative to uniform advance")
for p in ax:
    p.axhline(0,color="0.55",lw=.8); p.grid(alpha=.23); p.legend()
plt.show()

## Try a tooth-count specification
Find the smallest integer tooth count whose peak-to-peak speed variation is below 0.5% of the mean. Does increasing shaft speed change this percentage? Decide whether a sensor that counts only whole pitches would detect the position error drawn on the right.

In [ ]:
for z in range(4,200):
    a = np.pi/z
    ripple = a*np.tan(a/2)
    if ripple < .005:
        print(f"First tooth count below 0.5%: {z}; ripple = {100*ripple:.4f}%")
        break

## Check the mean with four velocity samples
Use a four-point Gauss rule in engagement phase. Each node has a different integration weight. OneSimulation supplies the quadrature table in a local installation. The browser computes equivalent Gauss nodes and weights with NumPy.

In [ ]:
try:
    from onesim.numerics.discretization.geometry.carriers.kernels.quadrature.rules.reference_quadrature import gauss_legendre
    nodes, weights = np.asarray(gauss_legendre(4)).T
    method = "OneSimulation Gauss quadrature"
except ImportError:
    nodes, weights = np.polynomial.legendre.leggauss(4)
    method = "NumPy Gauss quadrature"

# Map the reference interval [-1, 1] to one complete engagement.
gauss_phase = (nodes + 1) / 2
sampled_speed = chain(teeth, rpm, gauss_phase)[1]
gauss_mean = weights @ sampled_speed / 2
print(f"{method} with four samples: {gauss_mean:.8f} m/s")
assert np.isclose(gauss_mean, mean, rtol=1e-7)